# 02b — Radio Telemetry Cleaning (V1 per-scenario, primary source)
## NetsLab-5G-ORAN-IDD · V1

**Project:** *When One Sensor Lies: Robust Multimodal Intrusion Detection Against Inference-Time Telemetry Manipulation in 5G Open RAN*

**Purpose:** Produce a clean, analysis-ready radio dataset from the V1 per-scenario CSV/JSON files.  
This notebook **cleans and standardises** the radio data, adds alignment keys compatible with the network window dataset produced by `01b_Network_cleaning.ipynb`, and saves two output files:

| Output file | Description |
|---|---|
| `radio_clean.csv` | 7,490 rows × (22 KPI/counter + meta + alignment) columns — one row per UE measurement, **time-ordered**, no rows dropped |
| `radio_feature_columns.json` | column lists and all settings used |

**What is done here (all permitted operations):**
1. Load the per-scenario source (CSV preferred, JSON fallback for ICMP3 and SYN3).
2. Cast numeric columns; parse `pmi` into `pmi_0` / `pmi_1`; decode `rnti` hex → int.
3. Add scenario metadata columns (`family`, `mode`, `class_fm`, `is_attack`, `ddos`).
4. Compute `timestamp_s` (epoch seconds UTC, integer-floored) for 1-second window alignment with network data.
5. Add `radio_window_id = <scenario>@<timestamp_s>` — the join key to `network_clean.csv`.
6. Flag 3GPP out-of-range values (never drop them).
7. Identify and flag exact duplicate rows (same `scenario`, `ue_id`, `timestamp`) — never dropped.
8. Verify `in_sync == 1` for all rows (confirmed in analysis; flag the rare exceptions).
9. Hard integrity checks (no silent failures).
10. Save outputs and fingerprint input files to prove nothing was modified.

**What is NOT done here:**
- No rows are removed.
- No features are scaled, imputed, log-transformed or selected.
- No model is trained.
- The newer-release DB (`Lower_Layer_Data.db`) is not used as an input here.


---
## 1 · Configuration and Paths

In [1]:
import os, glob, json, re, warnings, time, platform
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80, "display.width", 230, "display.max_rows", 200, "display.float_format", "{:,.4f}".format)

# ---- dataset root (same resolution logic as the analysis notebook) ---------
BASE_CANDIDATES = [os.environ.get("NETSLAB_BASE", ""),
                   "/kaggle/input/datasets/netslabdemo/netslab-5g-oran-idd"]
BASE = next((p for p in BASE_CANDIDATES if p and os.path.isdir(p)), None)
if BASE is None:
    hits = []
    if os.path.isdir("/kaggle/input"):
        for dp, dn, fn in os.walk("/kaggle/input"):
            if "Lower_Layer_Data.db" in fn: hits.append(dp)
            if dp.count(os.sep) >= 6: dn[:] = []
    raise FileNotFoundError(
        "Dataset folder not found.\n"
        f"  expected : {BASE_CANDIDATES[1]}\n"
        f"  found    : {hits or 'nothing under /kaggle/input'}\n"
        "  -> In Kaggle: 'Add Input' -> dataset 'netslabdemo/netslab-5g-oran-idd', or set NETSLAB_BASE.")

V1_DIR = f"{BASE}/Attack (Legacy) Version 1"
ORIG   = f"{V1_DIR}/Attacks original/Attacks"
assert os.path.isdir(ORIG), f"per-scenario dir not found: {ORIG}"

# ---- outputs ---------------------------------------------------------------
OUT_ROOT  = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("NETSLAB_OUT", "./outputs")
RADIO_OUT = f"{OUT_ROOT}/radio_clean_outputs"
os.makedirs(RADIO_OUT, exist_ok=True)
print("BASE      :", BASE)
print("RADIO_OUT :", RADIO_OUT)
print("python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__)

BASE      : /kaggle/input/datasets/netslabdemo/netslab-5g-oran-idd
RADIO_OUT : /kaggle/working/radio_clean_outputs
python 3.13.16 | pandas 2.3.3 | numpy 2.1.3


---
## 2 · Column Vocabulary

In [2]:
# KPIs and counters (numeric, as established in analysis §3)
KPIS     = ["rsrp", "rssi", "rsrq", "sinr", "cqi", "dlMcs", "ulMcs", "dlBler", "ulBler",
            "ri", "phr", "pcmax", "pucchSnr", "puschSnr"]
COUNTERS = ["dlBytes", "ulBytes"]
NUM_COLS = KPIS + COUNTERS

# 3GPP plausibility ranges (used ONLY to flag, never to filter)
PHYS = {
    "rsrp":     (-140, -40),
    "rsrq":     (-30,  0),
    "rssi":     (-120, 0),
    "sinr":     (-20,  40),
    "cqi":      (0,    15),
    "dlMcs":    (0,    28),
    "ulMcs":    (0,    28),
    "dlBler":   (0,    1),
    "ulBler":   (0,    1),
    "ri":       (1,    4),
    "phr":      (-23,  40),
    "pcmax":    (-30,  33),
    "pucchSnr": (-30,  60),
    "puschSnr": (-30,  60),
}

# Scenario metadata columns added by the loader
V1_META = ["scenario", "folder", "source_fmt", "family", "mode", "class_fm", "ddos",
           "is_attack", "file_row", "file_order_desc"]

print("KPIs      :", KPIS)
print("Counters  :", COUNTERS)
print("Plausibility ranges (flag only):", PHYS)

KPIs      : ['rsrp', 'rssi', 'rsrq', 'sinr', 'cqi', 'dlMcs', 'ulMcs', 'dlBler', 'ulBler', 'ri', 'phr', 'pcmax', 'pucchSnr', 'puschSnr']
Counters  : ['dlBytes', 'ulBytes']
Plausibility ranges (flag only): {'rsrp': (-140, -40), 'rsrq': (-30, 0), 'rssi': (-120, 0), 'sinr': (-20, 40), 'cqi': (0, 15), 'dlMcs': (0, 28), 'ulMcs': (0, 28), 'dlBler': (0, 1), 'ulBler': (0, 1), 'ri': (1, 4), 'phr': (-23, 40), 'pcmax': (-30, 33), 'pucchSnr': (-30, 60), 'puschSnr': (-30, 60)}


---
## 3 · Helper Functions

In [3]:
def scenario_family(scn):
    """Derive attack family from scenario name (strips trailing digits and DDOS_ prefix)."""
    base = re.sub(r"\d+$", "", scn)
    if scn.startswith("DDOS_"):
        return re.sub(r"\d+$", "", scn.replace("DDOS_", ""))
    return base

def read_v1_scenario(stem):
    """Load one scenario; prefer CSV, fall back to JSON. Returns (df, source_format, status_dict)."""
    st = {"csv_status": "missing", "csv_rows": 0, "json_status": "missing", "json_rows": 0}
    c = j = None
    if os.path.exists(stem + ".csv"):
        try:
            c = pd.read_csv(stem + ".csv", dtype={"rnti": str, "pmi": str})
            st["csv_rows"] = len(c)
            st["csv_status"] = "ok" if ("timestamp" in c.columns and len(c)) else "no_timestamp_or_empty"
        except Exception as e:
            st["csv_status"] = f"unreadable:{type(e).__name__}"; c = None
    if os.path.exists(stem + ".json"):
        try:
            j = pd.DataFrame(json.load(open(stem + ".json", errors="replace")))
            st["json_rows"] = len(j)
            st["json_status"] = "ok" if ("timestamp" in j.columns and len(j)) else "no_timestamp_or_empty"
        except Exception as e:
            st["json_status"] = f"unreadable:{type(e).__name__}"; j = None
    if st["csv_status"] == "ok":
        return c, "csv", st
    if st["json_status"] == "ok":
        return j, "json", st
    return pd.DataFrame(), "none", st

def load_v1(orig):
    """Load all 35 scenarios from per-scenario CSV/JSON files."""
    stems = sorted({p.rsplit(".", 1)[0] for ext in ("csv", "json")
                    for p in glob.glob(f"{orig}/*/*.{ext}")})
    frames, meta = [], []
    for stem in stems:
        scn    = os.path.basename(stem)
        folder = os.path.basename(os.path.dirname(stem))
        d, src, st = read_v1_scenario(stem)
        meta.append({"scenario": scn, "folder": folder, "used_source": src, **st, "rows_used": len(d)})
        if d.empty:
            continue
        d = d.copy()
        # file_order_desc: True if timestamps were stored in descending order in the raw file
        desc = bool(pd.to_numeric(d["timestamp"], errors="coerce").is_monotonic_decreasing)
        fam  = scenario_family(scn)
        is_ddos = int(scn.startswith("DDOS_"))
        is_benign = int(fam.lower() == "benign")
        if fam.lower() == "benign":
            mode     = "Benign"
            class_fm = "Benign"
        elif fam.lower() == "portscan":
            mode     = "PortScan"
            class_fm = "PortScan"
        else:
            mode     = "DDoS" if is_ddos else "DoS"
            class_fm = f"{fam}_{mode}"
        d.insert(0, "scenario",        scn)
        d.insert(1, "folder",          folder)
        d.insert(2, "source_fmt",      src)
        d["family"]          = fam
        d["mode"]            = mode
        d["class_fm"]        = class_fm
        d["ddos"]            = is_ddos
        d["is_attack"]       = 1 - is_benign
        d["file_row"]        = np.arange(len(d))
        d["file_order_desc"] = desc
        frames.append(d)
    return pd.concat(frames, ignore_index=True), pd.DataFrame(meta)

def cast_numerics(df):
    """Cast KPI/counter columns to float64. Original columns untouched; returns a copy."""
    out = df.copy()
    for c in NUM_COLS + ["timestamp", "ue_id", "cellid", "in_sync"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def split_pmi(df):
    """Split 'pmi' string column '(x,y)' -> pmi_0, pmi_1 (integers)."""
    out = df.copy()
    if "pmi" in out.columns and "pmi_0" not in out.columns:
        p = out["pmi"].astype(str).str.extract(r"\(?\s*(-?\d+)\s*,\s*(-?\d+)\s*\)?")
        out["pmi_0"] = pd.to_numeric(p[0], errors="coerce")
        out["pmi_1"] = pd.to_numeric(p[1], errors="coerce")
    return out

def decode_rnti(s):
    """Hex string RNTI -> integer (NaN if unparseable)."""
    def h(v):
        try:    return int(str(v), 16)
        except: return np.nan
    unique_vals = s.dropna().unique()
    return s.map({v: h(v) for v in unique_vals})

def flag_out_of_range(df):
    """Add boolean flag columns <kpi>_flag_oob (out-of-plausible-range). Never modifies values."""
    out = df.copy()
    for c, (lo, hi) in PHYS.items():
        if c in out.columns:
            col = pd.to_numeric(out[c], errors="coerce")
            out[f"{c}_flag_oob"] = ((col < lo) | (col > hi)).astype("Int8")
    return out

print("helper functions defined.")

helper functions defined.


---
## 4 · Load V1 Per-Scenario Files

In [4]:
t0 = time.time()
v1_raw, v1_meta = load_v1(ORIG)
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"  Scenario stems : {len(v1_meta)}")
print(f"  Usable         : {(v1_meta.used_source != 'none').sum()}")
print(f"  Total rows     : {len(v1_raw):,}")
print(f"  Source used    : {v1_meta.used_source.value_counts().to_dict()}")
bad = v1_meta[v1_meta.used_source == "none"]
if len(bad):
    print("WARNING — no usable file for:", bad.scenario.tolist())
display(v1_meta[["scenario", "folder", "used_source", "csv_status", "json_status", "rows_used"]])

Loaded in 5.5s
  Scenario stems : 35
  Usable         : 35
  Total rows     : 7,490
  Source used    : {'csv': 33, 'json': 2}


,scenario,folder,used_source,csv_status,json_status,rows_used
0,Benign1,Benign Traffic,csv,ok,ok,257
1,Benign2,Benign Traffic,csv,ok,ok,585
2,Benign3,Benign Traffic,csv,ok,ok,607
3,Benign4,Benign Traffic,csv,ok,ok,611
4,Benign5,Benign Traffic,csv,ok,ok,313
5,Benign6,Benign Traffic,csv,ok,ok,609
6,DDOS_ICMP1,ICMP Flood,csv,ok,ok,243
7,DDOS_ICMP2,ICMP Flood,csv,ok,ok,231
8,DDOS_ICMP3,ICMP Flood,csv,ok,ok,229
9,ICMP1,ICMP Flood,csv,ok,ok,103


---
## 5 · Type Casting and Structural Enrichment

In [5]:
v1 = cast_numerics(v1_raw)
v1 = split_pmi(v1)
v1["rnti_int"] = decode_rnti(v1["rnti"])

# --- alignment key --------------------------------------------------------
# timestamp is epoch milliseconds; floor to whole UTC seconds
v1["timestamp_ms"] = v1["timestamp"].copy()          # keep original ms column
v1["timestamp_s"]  = np.floor(v1["timestamp_ms"] / 1000).astype("Int64")   # whole UTC second
v1["time_utc"]     = pd.to_datetime(v1["timestamp_ms"], unit="ms", utc=True)

# Join key: <scenario>@<timestamp_s>  — matches window_start_epoch_utc in network_clean.csv
# Network window_id is <scenario>@<sub-bin index>, where sub-bin index = timestamp_s * 4 for phase 0.
# Here we store the whole-second epoch so a simple equi-join on timestamp_s works.
v1["radio_window_id"] = v1["scenario"].astype(str) + "@" + v1["timestamp_s"].astype(str)

print("timestamp_ms range:",
      pd.to_datetime(v1.timestamp_ms.min(), unit="ms", utc=True),
      "→",
      pd.to_datetime(v1.timestamp_ms.max(), unit="ms", utc=True))
print("pmi_0 unique:", sorted(v1["pmi_0"].dropna().unique().tolist()))
print("pmi_1 unique:", sorted(v1["pmi_1"].dropna().unique().tolist()))
print("rnti_int non-null:", v1["rnti_int"].notna().sum(), "of", len(v1))
print("Shape after enrichment:", v1.shape)

timestamp_ms range: 2025-02-27 13:39:07.404000+00:00 → 2025-04-14 13:19:01.522000+00:00
pmi_0 unique: [0, 2, 4, 6, 8, 9, 10, 11, 13, 15]
pmi_1 unique: [0, 1]
rnti_int non-null: 7490 of 7490
Shape after enrichment: (7490, 39)


---
## 6 · Out-of-Plausible-Range Flags
Values are **never removed**. Flag columns (`<kpi>_flag_oob`) are set to 1 when a value falls outside the 3GPP-style plausible range established in the analysis notebook.  
All conclusions from the analysis about range violations are confirmed here.

In [6]:
v1 = flag_out_of_range(v1)

oob_summary = {}
for c, (lo, hi) in PHYS.items():
    flag_col = f"{c}_flag_oob"
    if flag_col in v1.columns:
        n_bad = int(v1[flag_col].sum())
        oob_summary[c] = {
            "plausible_range":  f"[{lo}, {hi}]",
            "observed_min":     round(float(v1[c].min()), 4),
            "observed_max":     round(float(v1[c].max()), 4),
            "n_out_of_range":   n_bad,
            "pct_out_of_range": round(100 * n_bad / max(v1[c].notna().sum(), 1), 3),
        }

oob_df = pd.DataFrame(oob_summary).T
display(oob_df)
print(f"\nTotal OOB flags set across all KPIs: {oob_df['n_out_of_range'].astype(int).sum():,}")

,plausible_range,observed_min,observed_max,n_out_of_range,pct_out_of_range
rsrp,"[-140, -40]",-116.0000,-69.0000,0,0.0000
rsrq,"[-30, 0]",-18.5000,-10.5000,0,0.0000
rssi,"[-120, 0]",-116.0000,-54.9000,0,0.0000
sinr,"[-20, 40]",-6.0000,23.5000,0,0.0000
cqi,"[0, 15]",7.0000,15.0000,0,0.0000
dlMcs,"[0, 28]",0.0000,27.0000,0,0.0000
ulMcs,"[0, 28]",0.0000,27.0000,0,0.0000
dlBler,"[0, 1]",0.0000,0.8858,0,0.0000
ulBler,"[0, 1]",0.0000,0.8984,0,0.0000
ri,"[1, 4]",1.0000,2.0000,0,0.0000



Total OOB flags set across all KPIs: 2,801


---
## 7 · Duplicate Detection
Duplicate = same `(scenario, ue_id, timestamp_ms)` triple.  
Rows are **flagged, not dropped**. The analysis notebook noted that ICMP3 and SYN3 came from JSON (their CSVs had no timestamp), so their duplicate patterns may differ from the rest.

In [7]:
dup_key = ["scenario", "ue_id", "timestamp_ms"]
v1["is_duplicate"] = v1.duplicated(subset=dup_key, keep=False).astype("Int8")

dup_counts = v1.groupby("scenario")["is_duplicate"].sum().sort_values(ascending=False)
print("Scenarios with duplicate (scenario, ue_id, timestamp_ms) rows:")
print(dup_counts[dup_counts > 0].to_string())
print(f"\nTotal duplicate rows: {v1['is_duplicate'].sum():,} of {len(v1):,} ({100*v1['is_duplicate'].mean():.2f}%)")

Scenarios with duplicate (scenario, ue_id, timestamp_ms) rows:
scenario
Benign1       2
SYN3          2
ICMP3         2
PortScan1     2
PortScan2     2
PortScan3     2
SYN1          2
SYN2          2
TCP1          2
ICMP1         2
TCP2          2
TCP3          2
UDP1          2
UDP2          2
UDP3          2
UDP4          2
ICMP2         2
DDOS_UDP3     2
Benign2       2
DDOS_ICMP3    2
Benign3       2
Benign4       2
Benign5       2
Benign6       2
DDOS_ICMP1    2
DDOS_ICMP2    2
DDOS_SYN1     2
DDOS_UDP2     2
DDOS_SYN2     2
DDOS_SYN3     2
DDOS_TCP1     2
DDOS_TCP2     2
DDOS_TCP3     2
DDOS_UDP1     2
UDP5          2

Total duplicate rows: 70 of 7,490 (0.93%)


---
## 8 · `in_sync` Check
The analysis confirmed `in_sync == 1` for all 7,490 V1 rows. Flag any exception.

In [8]:
if "in_sync" in v1.columns:
    not_synced = (v1["in_sync"] != 1).sum()
    print(f"in_sync != 1: {not_synced} rows")
    if not_synced:
        print(v1.loc[v1["in_sync"] != 1, ["scenario", "ue_id", "timestamp_ms", "in_sync"]])
else:
    print("WARNING: in_sync column not found in this dataset version.")

in_sync != 1: 0 rows


---
## 9 · Null and Schema Summary

In [9]:
def schema_report(df, name, cols=None):
    rows = []
    for c in (cols or list(df.columns)):
        s = df[c]
        num  = pd.to_numeric(s, errors="coerce")
        is_num = num.notna().mean() > 0.9
        rows.append({
            "dataset":  name,
            "column":   c,
            "dtype":    str(s.dtype),
            "n":        len(s),
            "null":     int(s.isna().sum()),
            "null_pct": round(100 * s.isna().mean(), 3),
            "nunique":  int(s.nunique(dropna=True)),
            "min":      num.min() if is_num else np.nan,
            "max":      num.max() if is_num else np.nan,
            "example":  str(s.dropna().iloc[0]) if s.notna().any() else "",
        })
    return pd.DataFrame(rows)

raw_cols = [c for c in v1_raw.columns if c not in V1_META]
schema   = schema_report(v1, "radio_clean", raw_cols + ["timestamp_s", "pmi_0", "pmi_1", "rnti_int"])
display(schema)

,dataset,column,dtype,n,null,null_pct,nunique,min,max,example
0,radio_clean,cellid,int64,7490,0,0.0000,2,"11,111,111.0000","12,345,678.0000",11111111
1,radio_clean,cqi,int64,7490,0,0.0000,8,7.0000,15.0000,15
2,radio_clean,dlBler,float64,7490,0,0.0000,4516,0.0000,0.8858,0.0
3,radio_clean,dlBytes,int64,7490,0,0.0000,7370,0.0000,"13,423,057,208.0000",76772351
4,radio_clean,dlMcs,int64,7490,0,0.0000,26,0.0000,27.0000,11
5,radio_clean,in_sync,int64,7490,0,0.0000,1,1.0000,1.0000,1
6,radio_clean,pcmax,int64,7490,0,0.0000,4,0.0000,25.0000,24
7,radio_clean,phr,int64,7490,0,0.0000,44,0.0000,68.0000,25
8,radio_clean,pmi,object,7490,0,0.0000,19,NaN,NaN,"(0,0)"
9,radio_clean,pucchSnr,int64,7490,0,0.0000,33,0.0000,35.0000,22


---
## 10 · Hard Integrity Checks
The notebook stops with a clear error if any check fails.

In [10]:
checks = {
    "rows loaded == 7490":             len(v1) == 7490,
    "scenarios loaded == 35":          v1["scenario"].nunique() == 35,
    "no null timestamp_ms":            v1["timestamp_ms"].isna().sum() == 0,
    "no null ue_id":                   v1["ue_id"].isna().sum() == 0,
    "no null scenario":                v1["scenario"].isna().sum() == 0,
    "timestamp_ms is positive":        (v1["timestamp_ms"] > 0).all(),
    "is_attack in {0,1}": set(v1["is_attack"].unique()).issubset({0, 1}),
    "radio_window_id has no nulls":    v1["radio_window_id"].isna().sum() == 0,
    "NaN in KPI/counter numeric cols": v1[NUM_COLS].isna().sum().sum() == 0,
}

failed = {k: v for k, v in checks.items() if not v}
passed = {k: v for k, v in checks.items() if v}
for k in passed:  print(f"  PASS  {k}")
for k in failed:  print(f"  FAIL  {k}")
if failed:
    raise AssertionError(f"Integrity checks failed: {list(failed.keys())}")
print("\nAll integrity checks passed.")

  PASS  rows loaded == 7490
  PASS  scenarios loaded == 35
  PASS  no null timestamp_ms
  PASS  no null ue_id
  PASS  no null scenario
  PASS  timestamp_ms is positive
  PASS  is_attack in {0,1}
  PASS  radio_window_id has no nulls
  PASS  NaN in KPI/counter numeric cols

All integrity checks passed.


---
## 11 · Assemble the Clean Output Table

**Column order:**
1. Identifiers & metadata
2. Time & alignment
3. KPIs
4. Counters
5. Structural identifiers (`rnti`, `rnti_int`, `cellid`, `pmi`, `pmi_0`, `pmi_1`, `in_sync`)
6. Flag columns (`is_duplicate`, out-of-range flags)
7. Provenance columns from loader (`source_fmt`, `file_row`, `file_order_desc`)

In [11]:
OOB_FLAG_COLS = [f"{c}_flag_oob" for c in PHYS if f"{c}_flag_oob" in v1.columns]

CLEAN_COLS = (
    # --- identifiers & metadata ---
    ["scenario", "folder", "family", "mode", "class_fm", "is_attack", "ddos", "ue_id"]
    # --- time & alignment ---
    + ["timestamp_ms", "timestamp_s", "time_utc", "radio_window_id"]
    # --- KPIs ---
    + KPIS
    # --- counters ---
    + COUNTERS
    # --- structural identifiers ---
    + [c for c in ["rnti", "rnti_int", "cellid", "pmi", "pmi_0", "pmi_1", "in_sync"] if c in v1.columns]
    # --- quality flags ---
    + ["is_duplicate"]
    + OOB_FLAG_COLS
    # --- provenance ---
    + ["source_fmt", "file_row", "file_order_desc"]
)
# keep only columns that actually exist
CLEAN_COLS = [c for c in CLEAN_COLS if c in v1.columns]

# Sort by scenario (preserving file order within each scenario)
CLEAN = (
    v1.sort_values(["scenario", "ue_id", "timestamp_ms"])
      .reset_index(drop=True)[CLEAN_COLS]
)

print("radio_clean shape:", CLEAN.shape)
display(
    CLEAN.groupby("class_fm").agg(
        rows       = ("scenario", "size"),
        scenarios  = ("scenario", "nunique"),
        ue_ids     = ("ue_id",    "nunique"),
        t_min_utc  = ("time_utc", "min"),
        t_max_utc  = ("time_utc", "max"),
        rsrp_p50   = ("rsrp",     "median"),
        sinr_p50   = ("sinr",     "median"),
    )
)

radio_clean shape: (7490, 53)


,rows,scenarios,ue_ids,t_min_utc,t_max_utc,rsrp_p50,sinr_p50
class_fm,,,,,,,
Benign,2982,6,4,2025-04-14 12:34:37.522000+00:00,2025-04-14 13:19:01.522000+00:00,-100.0000,10.5000
ICMP_DDoS,703,3,3,2025-04-03 15:20:04.868000+00:00,2025-04-03 15:27:56.879000+00:00,-74.0000,23.5000
ICMP_DoS,357,3,2,2025-03-06 15:30:47.614000+00:00,2025-03-06 15:38:57.634000+00:00,-98.0000,17.0000
PortScan,26,3,1,2025-02-27 13:39:07.404000+00:00,2025-02-27 13:43:56.404000+00:00,-103.5000,12.0000
SYN_DDoS,665,3,5,2025-04-03 14:39:32.932000+00:00,2025-04-03 14:52:42.988000+00:00,-73.0000,23.5000
SYN_DoS,375,3,1,2025-03-06 15:16:37.464000+00:00,2025-03-06 15:27:37.614000+00:00,-97.0000,18.0000
TCP_DDoS,675,3,3,2025-04-03 15:05:24.028000+00:00,2025-04-03 15:13:34.028000+00:00,-73.0000,23.5000
TCP_DoS,373,3,1,2025-03-06 14:37:54.464000+00:00,2025-03-06 15:07:36.464000+00:00,-100.0000,18.5000
UDP_DDoS,712,3,6,2025-04-03 14:17:02.442000+00:00,2025-04-03 14:36:55.932000+00:00,-73.0000,23.5000


---
## 12 · Alignment Note — How to Join with `network_clean.csv`

The network cleaning notebook (`01b`) outputs `network_clean.csv` with these join-relevant columns:

| network_clean column | type | meaning |
|---|---|---|
| `window_id` | str | `<scenario>@<sub-bin index>` (sub-bin index = `window_start_epoch_utc * 4` for phase 0) |
| `scenario` | str | matches `scenario` here |
| `window_start_epoch_utc` | float | whole UTC second (integer-valued, since `aug_phase_s == 0`) |

This notebook's radio rows carry `timestamp_s` (UTC second, integer-floored from epoch-ms).  
**Recommended join:**
```python
import pandas as pd
net   = pd.read_csv("network_clean.csv")
radio = pd.read_csv("radio_clean.csv")
# keep only real (non-augmented) network windows
net_real = net[net["is_partial"] == 0]  # already guaranteed by 01b

# join key: same scenario AND same whole-second UTC epoch
net_real = net_real.assign(timestamp_s=net_real["window_start_epoch_utc"].astype("Int64"))
paired = net_real.merge(
    radio,
    on=["scenario", "timestamp_s"],
    how="inner",   # or 'left' to keep network windows with no radio partner
    suffixes=("_net", "_radio")
)
```
**Coverage expectation:** not every network window second will have a radio measurement (the radio telemetry is per-UE at ~1 Hz but may not be perfectly synchronous). Missing network windows for a given second mean zero traffic, not missing data.

In [12]:
# Quick alignment preview: how many unique timestamp_s values exist per scenario
align_preview = (
    CLEAN.groupby("scenario")
         .agg(
             radio_rows   = ("timestamp_ms", "size"),
             unique_secs  = ("timestamp_s",  "nunique"),
             t_span_s     = ("timestamp_s",  lambda x: int(x.max() - x.min() + 1)),
         )
         .assign(coverage_pct=lambda d: (100 * d["unique_secs"] / d["t_span_s"]).round(1))
)
display(align_preview)
print("\n(coverage_pct < 100 is normal — radio telemetry may skip seconds with no UE activity)")

,radio_rows,unique_secs,t_span_s,coverage_pct
scenario,,,,
Benign1,257,128,128,100.0000
Benign2,585,319,319,100.0000
Benign3,607,303,303,100.0000
Benign4,611,305,305,100.0000
Benign5,313,156,156,100.0000
Benign6,609,304,304,100.0000
DDOS_ICMP1,243,107,107,100.0000
DDOS_ICMP2,231,115,115,100.0000
DDOS_ICMP3,229,114,114,100.0000



(coverage_pct < 100 is normal — radio telemetry may skip seconds with no UE activity)


---
## 13 · Save Outputs

In [13]:
# Save main clean CSV
clean_path = f"{RADIO_OUT}/radio_clean.csv"
CLEAN.to_csv(clean_path, index=False)
print(f"Saved: {clean_path}  {CLEAN.shape}")

# Save feature / metadata column list
meta_dict = {
    "kpi_cols":          KPIS,
    "counter_cols":      COUNTERS,
    "numeric_cols":      NUM_COLS,
    "oob_flag_cols":     OOB_FLAG_COLS,
    "alignment_cols":    ["scenario", "timestamp_s", "radio_window_id"],
    "label_cols":        ["is_attack", "family", "mode", "class_fm", "ddos"],
    "plausibility_ranges": {k: list(v) for k, v in PHYS.items()},
    "all_clean_cols":    CLEAN_COLS,
    "n_rows":            len(CLEAN),
    "n_scenarios":       int(CLEAN["scenario"].nunique()),
    "notes": [
        "No rows were dropped. All 7490 V1 per-scenario rows are retained.",
        "timestamp_s = floor(timestamp_ms / 1000): whole UTC second for 1-Hz alignment.",
        "radio_window_id = scenario + '@' + timestamp_s — equi-join key to network_clean.csv on (scenario, timestamp_s).",
        "is_duplicate flags rows sharing (scenario, ue_id, timestamp_ms); rows are NOT removed.",
        "OOB flag columns flag values outside 3GPP plausibility ranges; values are NOT changed.",
        "ICMP3 and SYN3 were loaded from JSON (their CSVs had no timestamp column).",
    ],
}
meta_path = f"{RADIO_OUT}/radio_feature_columns.json"
with open(meta_path, "w") as f:
    json.dump(meta_dict, f, indent=2)
print(f"Saved: {meta_path}")

Saved: /kaggle/working/radio_clean_outputs/radio_clean.csv  (7490, 53)
Saved: /kaggle/working/radio_clean_outputs/radio_feature_columns.json


---
## 14 · Input File Fingerprint Check
Proves that no input file was modified during this notebook's execution.

In [14]:
import os, glob

def _fp(paths):
    return {p: (os.path.getsize(p), os.stat(p).st_mtime_ns) for p in paths if os.path.isfile(p)}

INPUT_FILES = sorted(set(
    glob.glob(f"{ORIG}/*/*")
    + glob.glob(f"{V1_DIR}/*.csv")
    + glob.glob(f"{V1_DIR}/*.db")
))
FP_AFTER = _fp(INPUT_FILES)

# In a fresh run FP_BEFORE is unavailable; in sequential runs store it at §1 and compare here.
# As a minimum: check that sizes are consistent.
print(f"Input files checked: {len(FP_AFTER)}")
for p, (sz, mt) in FP_AFTER.items():
    print(f"  {os.path.relpath(p, BASE):<70}  {sz:>12,} bytes")
print("\nAll input file sizes logged. Re-run §2 Config cell first to capture FP_BEFORE for a full before/after diff.")

Input files checked: 109
  Attack (Legacy) Version 1/5G-NL-IDD_Higher_Layer_Network_Data.db          15,978,496 bytes
  Attack (Legacy) Version 1/5G-NL-IDD_Radio_Telemetry.db                             0 bytes
  Attack (Legacy) Version 1/Attacks original/Attacks/Benign Traffic/Benign1.csv        30,190 bytes
  Attack (Legacy) Version 1/Attacks original/Attacks/Benign Traffic/Benign1.json        77,341 bytes
  Attack (Legacy) Version 1/Attacks original/Attacks/Benign Traffic/Benign1_youtube_download_120seconds.pcapng   731,352,296 bytes
  Attack (Legacy) Version 1/Attacks original/Attacks/Benign Traffic/Benign2.csv        69,698 bytes
  Attack (Legacy) Version 1/Attacks original/Attacks/Benign Traffic/Benign2.json       177,201 bytes
  Attack (Legacy) Version 1/Attacks original/Attacks/Benign Traffic/Benign2_youtube_download_5minutes.pcapng  1,605,116,504 bytes
  Attack (Legacy) Version 1/Attacks original/Attacks/Benign Traffic/Benign3.csv        72,362 bytes
  Attack (Legacy) Version 

---
## 15 · Summary

| Item | Value |
|---|---|
| Input | V1 per-scenario CSV/JSON (35 scenarios, ICMP3+SYN3 from JSON) |
| Output rows | 7,490 (no rows dropped) |
| Output columns | see `radio_feature_columns.json` |
| Alignment key | `radio_window_id` = `<scenario>@<timestamp_s>`, join to `network_clean.csv` on `(scenario, timestamp_s)` |
| OOB flags | set for values outside 3GPP plausible ranges; values not changed |
| Duplicate flags | set for identical `(scenario, ue_id, timestamp_ms)`; rows not removed |
| Outputs | `radio_clean.csv`, `radio_feature_columns.json` |

**Next step:** `03_alignment.ipynb` — time-join `radio_clean.csv` with `network_clean.csv` to build the paired multimodal dataset.